# Multi-Facet Vector Search Indices

Creates 3 separate Vector Search indices on `fichas_vs_gold_v2`, each embedding a different forensic facet (tatuajes, vestimenta, señas). Sexo is used as a hard blocking filter — not embedded — so the semantic signal stays clean. Each index returns its own similarity score, enabling weighted re-ranking with forensic importance: tatuajes (0.65), vestimenta (0.25), señas (0.10).

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    VectorIndexType,
    PipelineType,
)
import time, json

w = WorkspaceClient()

ENDPOINT_NAME = "forensic"
GOLD_TABLE = "proyecto_mna.fichas_busqueda.fichas_vs_gold_v2"
EMBEDDING_MODEL = "databricks-qwen3-embedding-0-6b"
PRIMARY_KEY = "id_ficha"

# The 3 facet indices to create
FACET_INDICES = [
    {
        "name": "proyecto_mna.fichas_busqueda.vs_tatuajes_idx",
        "embedding_col": "texto_tatuajes",
        "description": "Tatuajes — most discriminative, weight 0.65",
    },
    {
        "name": "proyecto_mna.fichas_busqueda.vs_vestimenta_idx",
        "embedding_col": "texto_vestimenta",
        "description": "Vestimenta — high coverage but noisy, weight 0.25",
    },
    {
        "name": "proyecto_mna.fichas_busqueda.vs_senas_idx",
        "embedding_col": "texto_senas",
        "description": "Señas/cicatrices — low coverage, weight 0.10",
    },
]

print(f"Endpoint: {ENDPOINT_NAME}")
print(f"Source table: {GOLD_TABLE}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Indices to create: {len(FACET_INDICES)}")

In [0]:
# Columns to sync alongside embeddings (for blocking filters and re-ranking)
COLUMNS_TO_SYNC = [
    "id_ficha",
    "filter_sexo",
    "filter_grupo_edad",
    "filter_estado",
    "edad_desaparicion",
    "estatura_cm",
    "texto_tatuajes",
    "texto_vestimenta",
    "texto_senas",
]

created_indices = []
for facet in FACET_INDICES:
    index_name = facet["name"]
    embedding_col = facet["embedding_col"]
    
    # Check if index already exists
    existing = list(w.vector_search_indexes.list_indexes(endpoint_name=ENDPOINT_NAME))
    if any(idx.name == index_name for idx in existing):
        print(f"⏭️  Index {index_name} already exists, skipping")
        created_indices.append(index_name)
        continue
    
    print(f"Creating index: {index_name} (embedding on '{embedding_col}')")
    
    spec = DeltaSyncVectorIndexSpecRequest(
        source_table=GOLD_TABLE,
        pipeline_type=PipelineType.TRIGGERED,
        embedding_source_columns=[
            EmbeddingSourceColumn(
                name=embedding_col,
                embedding_model_endpoint_name=EMBEDDING_MODEL,
            )
        ],
        columns_to_sync=COLUMNS_TO_SYNC,
    )
    
    idx = w.vector_search_indexes.create_index(
        name=index_name,
        endpoint_name=ENDPOINT_NAME,
        primary_key=PRIMARY_KEY,
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=spec,
    )
    print(f"  ✅ Created: {idx.name}")
    created_indices.append(index_name)

print(f"\nTotal indices configured: {len(created_indices)}")

In [0]:
print("Waiting for indices to come ONLINE...\n")

for facet in FACET_INDICES:
    index_name = facet["name"]
    while True:
        idx = w.vector_search_indexes.get_index(index_name)
        status = idx.status
        ready = status.ready if status else None
        # Print available status attributes for debugging
        status_attrs = {k: v for k, v in status.as_dict().items() if v is not None} if status else {}
        print(f"  {index_name}: ready={ready}, status={status_attrs}")
        if ready:
            print(f"  ✅ {index_name} is ONLINE!\n")
            break
        time.sleep(15)
    
print("All indices are ready!")

In [0]:
# Test query: a necropsia tattoo description
QUERY_TEXT = "tatuaje monocromático a tinta negra con la figura de una cruz, localizado en región esternal"
FILTERS = {"filter_sexo": "HOMBRE"}

print(f"Query: {QUERY_TEXT}")
print(f"Blocking filter: {FILTERS}\n")

# Fix: define FACET_INDICES and test only the tatuajes index
FACET_INDICES = [
    {
        "name": "proyecto_mna.fichas_busqueda.vs_tatuajes_idx",
        "embedding_col": "texto_tatuajes",
        "description": "Tatuajes — most discriminative, weight 0.65",
    },
]

QUERY_TEXT = "tatuaje monocromático a tinta negra con la figura de una cruz, localizado en región esternal"
FILTERS = {"filter_sexo": "HOMBRE"}

print(f"Query: {QUERY_TEXT}")
print(f"Blocking filter: {FILTERS}\n")

for facet in FACET_INDICES:
    index_name = facet["name"]
    embedding_col = facet["embedding_col"]
    # Query only the tatuajes index
    print(f"--- {index_name} (embedding: {embedding_col}) ---")
    try:
        results = w.vector_search_indexes.query_index(
            index_name=index_name,
            columns=[
                "id_ficha", "filter_sexo", "filter_grupo_edad", "filter_estado",
                "edad_desaparicion", "texto_tatuajes", "texto_vestimenta",
            ],
            query_text=QUERY_TEXT,
            num_results=5,
            filters_json=json.dumps(FILTERS),
        )
        if results.result and results.result.data:
            for row in results.result.data:
                score = row[0] if len(row) > 0 else "?"
                print(f"  score={score:.4f}  id={row[1] if len(row) > 1 else '?'}  edad={row[5] if len(row) > 5 else '?'}  tatuajes={str(row[6])[:80] if len(row) > 6 else '?'}...")
        else:
            print("  No results")
    except Exception as e:
        print(f"  Error: {e}")
    print()

print("Test complete!")

In [0]:
# Test vestimenta index with a clothing description from a necropsia
VESTIMENTA_QUERY = "pantalon de mezclilla color azul, playera color gris, tenis color negro"
FILTERS_V = {"filter_sexo": "HOMBRE"}

print(f"Vestimenta query: {VESTIMENTA_QUERY}")
print(f"Blocking filter: {FILTERS_V}\n")

results_v = w.vector_search_indexes.query_index(
    index_name="proyecto_mna.fichas_busqueda.vs_vestimenta_idx",
    columns=["id_ficha", "filter_sexo", "filter_grupo_edad", "edad_desaparicion", "texto_vestimenta"],
    query_text=VESTIMENTA_QUERY,
    num_results=5,
    filters_json=json.dumps(FILTERS_V),
)

if results_v.result and results_v.result.data:
    for row in results_v.result.data:
        score = row[0]
        print(f"  score={score:.4f}  id={row[1]}  edad={row[4]}  vestimenta={str(row[5])[:100]}...")
else:
    print("  No results")